# TissueMC: GPU rung R1 on a Colab T4

Rung R1 accumulates absorbed weight per layer inside each thread and publishes it once per photon, instead of two global atomic adds per absorption event.

This notebook validates R1, compares it with the CPU on identical photons, benchmarks **R0, then R1, then R0 again in the same session** (so the GPU's own drift is visible), and reads the hardware counters for R0 and R1 with Nsight Compute. It ends with **one summary block** for you to paste back.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. Have your **R1 prediction recorded in `LOG.md`** (tell the assistant) before you run the benchmark cells.

Then *Runtime → Run all*. It takes roughly 15 to 25 minutes. Nothing here needs passwords or keys.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
!git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git
%cd tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -3
!./build/tissuemc --version

In [ ]:
# 1. Validate rung 1 against the published values and the independent oracle.
!tissuemc-validate --binary build/tissuemc --engine cuda --rung 1 --report results/gpu-r1-validation.json 2>&1 | grep -v '^  note' | tail -45

In [ ]:
# 2. Same photons through the CPU (float32) and rung 1: they should agree far more closely than noise.
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 1 --photons 2000000 --report results/gpu-r1-cross-engine.json

In [ ]:
# 3. Benchmarks, back to back in this session: R0, R1, then R0 again to show session drift.
!tissuemc-bench --engine cuda --rung 0 --binary build/tissuemc --output results/gpu-r0-session.json 2>&1 | tail -4
!tissuemc-bench --engine cuda --rung 1 --binary build/tissuemc --output results/gpu-r1.json 2>&1 | tail -4
!tissuemc-bench --engine cuda --rung 0 --binary build/tissuemc --output results/gpu-r0-again.json 2>&1 | tail -4

In [ ]:
# 4. Hardware counters (Nsight Compute) for both rungs: what actually limits each kernel.
!tissuemc-profile --rung 0 --binary build/tissuemc --output results/gpu-r0-profile.json
!tissuemc-profile --rung 1 --binary build/tissuemc --output results/gpu-r1-profile.json

In [ ]:
import json
import pathlib
import subprocess
import sys

# This notebook session started before `pip install -e .`, so it cannot see the package yet.
sys.path.insert(0, "python")
from tissuemc_analysis.ncu_profile import KEY_METRICS  # noqa: E402


def load(name):
    p = pathlib.Path("results") / name
    return json.loads(p.read_text()) if p.exists() else None


def bench_summary(b):
    if not b:
        return None
    return {
        "median_kernel_s": round(b["kernel"]["median"], 3),
        "kernel_seconds": [round(x, 3) for x in b["kernel_seconds"]],
        "million_events_per_s": round(b["events_per_second"] / 1e6, 1),
        "validated_first": b["validated_before_measuring"],
    }


def profile_summary(p):
    if not p:
        return None
    out = {}
    for section, metric, _ in KEY_METRICS:
        entry = p["sections"].get(section, {}).get(metric)
        if entry:
            out[metric] = f"{entry['value']} {entry['unit']}".strip()
    return out


val, cross = load("gpu-r1-validation.json"), load("gpu-r1-cross-engine.json")
commit = subprocess.run(
    ["git", "log", "-1", "--format=%H"], capture_output=True, text=True
).stdout.strip()
r1 = load("gpu-r1.json")
summary = {
    "commit": commit,
    "device": r1["device"] if r1 else None,
    "validation_r1": None
    if not val
    else {
        "checks": len(val),
        "passed": sum(r["passed"] for r in val),
        "failed": [
            f"{r['case']} {r['quantity']} z={r['z']:+.2f} ({r['source']})"
            for r in val
            if not r["passed"]
        ],
    },
    "cross_engine_r1": None
    if not cross
    else {
        "agree": cross["agree"],
        "gaps": {g["name"]: [g["value"], g["limit"]] for g in cross["gaps"]},
    },
    "benchmark": {
        "r0_first": bench_summary(load("gpu-r0-session.json")),
        "r1": bench_summary(r1),
        "r0_again": bench_summary(load("gpu-r0-again.json")),
    },
    "counters": {
        "r0": profile_summary(load("gpu-r0-profile.json")),
        "r1": profile_summary(load("gpu-r1-profile.json")),
    },
}
print("===== R1 SUMMARY =====")
print(json.dumps(summary, indent=1))